# Analisis PLS-SEM: Inovasi & Kinerja UMKM

Notebook ini mendemonstrasikan **Partial Least Squares SEM (PLS-SEM)** menggunakan package `seminr` di R.

### Dasar Teori

- **Resource-Based View** (Barney, 1991) — Kapabilitas inovasi sebagai sumber daya strategis
- **Market Orientation** (Narver & Slater, 1990) — Orientasi pasar mendorong keunggulan bersaing
- **Entrepreneurial Orientation** (Lumpkin & Dess, 1996) — Proaktivitas dan inovasi

### Model Penelitian

| Konstruk | Kode | Mode | Indikator |
|---|---|---|---|
| Kapabilitas Inovasi | IC | **Formatif (B)** | IC1–IC4 |
| Orientasi Pasar | MO | **Formatif (B)** | MO1–MO3 |
| Orientasi Kewirausahaan | EO | **Formatif (B)** | EO1–EO3 |
| Keunggulan Bersaing | CA | Reflektif (A) | CA1–CA4 |
| Kinerja Perusahaan | FP | Reflektif (A) | FP1–FP4 |

### Hipotesis

| # | Hipotesis | Path |
|---|---|---|
| H1 | Kapabilitas Inovasi → Keunggulan Bersaing (+) | IC → CA |
| H2 | Orientasi Pasar → Keunggulan Bersaing (+) | MO → CA |
| H3 | Orientasi Kewirausahaan → Keunggulan Bersaing (+) | EO → CA |
| H4 | Keunggulan Bersaing → Kinerja Perusahaan (+) | CA → FP |
| H5 | Kapabilitas Inovasi → Kinerja Perusahaan (+) | IC → FP |

In [1]:
library(seminr)
library(dplyr)


Attaching package: ‘dplyr’

The following objects are masked from ‘package:stats’:

    filter, lag

The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union



## 1. Import & Eksplorasi Data

In [2]:
dat <- read.csv("data/sem_innovation.csv")
glimpse(dat)

Rows: 200
Columns: 22
$ id        <int> 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 1…
$ IC1       <int> 5, 3, 5, 4, 4, 1, 4, 3, 5, 4, 4, 4, 4, 4, 1, 6, 5, 5, 4, 5, …
$ IC2       <int> 4, 5, 5, 3, 4, 7, 4, 4, 5, 7, 4, 6, 4, 4, 4, 6, 5, 6, 6, 7, …
$ IC3       <int> 6, 6, 4, 5, 4, 3, 7, 7, 5, 4, 6, 4, 6, 6, 5, 5, 6, 6, 5, 5, …
$ IC4       <int> 6, 4, 4, 5, 4, 4, 4, 4, 3, 5, 5, 3, 7, 4, 4, 2, 4, 3, 5, 5, …
$ MO1       <int> 3, 4, 4, 4, 6, 5, 6, 5, 5, 7, 4, 5, 7, 5, 7, 3, 5, 4, 3, 4, …
$ MO2       <int> 7, 6, 6, 4, 6, 4, 5, 5, 4, 5, 6, 5, 3, 3, 4, 3, 5, 4, 4, 2, …
$ MO3       <int> 5, 6, 5, 4, 6, 3, 6, 5, 2, 4, 6, 2, 6, 6, 6, 5, 4, 3, 5, 4, …
$ EO1       <int> 7, 2, 3, 6, 6, 4, 3, 3, 6, 2, 4, 5, 4, 5, 7, 4, 5, 4, 5, 4, …
$ EO2       <int> 3, 6, 4, 6, 4, 6, 4, 4, 5, 3, 4, 4, 6, 3, 6, 4, 4, 6, 5, 6, …
$ EO3       <int> 3, 4, 5, 4, 6, 5, 4, 4, 4, 4, 6, 4, 6, 5, 3, 4, 3, 7, 5, 5, …
$ CA1       <int> 4, 4, 4, 4, 3, 4, 4, 3, 4, 4, 4, 4, 4, 4, 5, 4, 3, 5, 3, 4, …
$ CA2       <int> 

In [3]:
# Statistik deskriptif indikator
dat |>
  select(IC1:FP4) |>
  tidyr::pivot_longer(everything(), names_to = "item", values_to = "skor") |>
  summarize(M = mean(skor), SD = sd(skor), .by = item) |>
  arrange(item)

# A tibble: 18 × 3
   item      M    SD
   <chr> <dbl> <dbl>
 1 CA1    4.06 0.703
 2 CA2    3.94 0.784
 3 CA3    3.86 0.806
 4 CA4    3.89 0.788
 5 EO1    4.62 1.21 
 6 EO2    4.46 1.28 
 7 EO3    4.77 0.917
 8 FP1    3.96 0.701
 9 FP2    4.00 0.793
10 FP3    4.05 0.781
11 FP4    4.04 0.811
12 IC1    4.48 1.22 
13 IC2    4.79 1.09 
14 IC3    5.11 1.08 
15 IC4    4.22 1.26 
16 MO1    4.93 1.07 
17 MO2    4.59 1.21 
18 MO3    4.56 1.15 

## 2. Spesifikasi & Estimasi Model

PLS-SEM membutuhkan dua komponen:
1. **Model Pengukuran** — hubungan indikator ↔ konstruk
2. **Model Struktural** — hubungan antar konstruk

Perhatikan perbedaan:
- **Mode A (Reflektif)**: Konstruk *mempengaruhi* indikator (panah dari konstruk ke indikator)
- **Mode B (Formatif)**: Indikator *membentuk* konstruk (panah dari indikator ke konstruk)

In [4]:
# --- Model Pengukuran ---
mm <- constructs(
  composite("IC", multi_items("IC", 1:4), weights = mode_B),  # Formatif
  composite("MO", multi_items("MO", 1:3), weights = mode_B),  # Formatif
  composite("EO", multi_items("EO", 1:3), weights = mode_B),  # Formatif
  composite("CA", multi_items("CA", 1:4), weights = mode_A),  # Reflektif
  composite("FP", multi_items("FP", 1:4), weights = mode_A)   # Reflektif
)

# --- Model Struktural ---
sm <- relationships(
  paths(from = c("IC", "MO", "EO"), to = "CA"),
  paths(from = c("CA", "IC"), to = "FP")
)

# --- Estimasi PLS ---
pls_model <- estimate_pls(
  data = dat,
  measurement_model = mm,
  structural_model = sm
)

model_sum <- summary(pls_model)

Generating the seminr model
All 200 observations are valid.


## 3. Evaluasi Model Pengukuran

### Konstruk Reflektif (Mode A): CA & FP

| Kriteria | Cut-off |
|---|---|
| Outer Loadings | ≥ 0.70 |
| Composite Reliability (rho_C) | ≥ 0.70 |
| AVE | ≥ 0.50 |
| HTMT | < 0.90 |

### Konstruk Formatif (Mode B): IC, MO, EO

| Kriteria | Cut-off |
|---|---|
| Outer Weights | Signifikan (p < 0.05 via bootstrap) |
| VIF | < 5 (tidak ada multikolinearitas) |

In [5]:
# Loadings (konstruk reflektif: CA, FP)
model_sum$loadings

        IC     MO     EO     CA     FP
IC1  0.208 -0.000 -0.000  0.000  0.000
IC2  0.595  0.000 -0.000  0.000  0.000
IC3  0.463  0.000 -0.000  0.000  0.000
IC4  0.571  0.000  0.000  0.000  0.000
MO1  0.000  0.661  0.000  0.000  0.000
MO2 -0.000  0.371  0.000  0.000 -0.000
MO3 -0.000  0.751  0.000  0.000  0.000
EO1 -0.000  0.000  0.914  0.000  0.000
EO2 -0.000  0.000  0.436  0.000  0.000
EO3  0.000 -0.000 -0.001 -0.000  0.000
CA1  0.000  0.000  0.000  0.644  0.000
CA2  0.000  0.000  0.000  0.784  0.000
CA3  0.000  0.000  0.000  0.609  0.000
CA4 -0.000  0.000  0.000  0.539  0.000
FP1  0.000  0.000  0.000  0.000  0.623
FP2  0.000  0.000  0.000  0.000  0.655
FP3  0.000  0.000  0.000  0.000  0.677
FP4  0.000  0.000  0.000  0.000  0.560

In [6]:
# Reliabilitas konstruk
model_sum$reliability

    alpha  rhoC   AVE  rhoA
IC -0.108 0.524 0.234 1.000
MO  0.180 0.631 0.380 1.000
EO  0.090 0.479 0.342 1.000
CA  0.547 0.742 0.423 0.578
FP  0.493 0.724 0.397 0.498

Alpha, rhoC, and rhoA should exceed 0.7 while AVE should exceed 0.5

In [7]:
# Weights (konstruk formatif: IC, MO, EO)
model_sum$weights

# VIF indikator (multikolinearitas)
model_sum$validity$vif_items

IC :
  IC1   IC2   IC3   IC4 
1.015 1.007 1.005 1.011 

MO :
  MO1   MO2   MO3 
1.012 1.005 1.015 

EO :
  EO1   EO2   EO3 
1.002 1.002 1.002 

CA :
  CA1   CA2   CA3   CA4 
1.183 1.206 1.124 1.147 

FP :
  FP1   FP2   FP3   FP4 
1.097 1.098 1.118 1.068 


## 4. Evaluasi Model Struktural

| Kriteria | Interpretasi |
|---|---|
| Path Coefficient (β) | Kekuatan & arah hubungan |
| t-value > 1.96 | Signifikan pada α = 0.05 |
| R² | 0.25 = lemah, 0.50 = moderat, 0.75 = kuat |
| f² | 0.02 = kecil, 0.15 = sedang, 0.35 = besar |

In [8]:
# Bootstrapping (1000 resamples)
set.seed(2026)
boot_model <- bootstrap_model(pls_model, nboot = 1000)
boot_sum <- summary(boot_model)

# Path coefficients + signifikansi
boot_sum$bootstrapped_paths

Bootstrapping model using seminr...
SEMinR Model successfully bootstrapped


           Original Est. Bootstrap Mean Bootstrap SD T Stat. 2.5% CI 97.5% CI
IC  ->  CA         0.120          0.135        0.081   1.474  -0.040    0.279
IC  ->  FP         0.175          0.192        0.101   1.726  -0.040    0.370
MO  ->  CA         0.364          0.373        0.061   5.996   0.251    0.486
EO  ->  CA         0.129          0.152        0.070   1.859   0.013    0.283
CA  ->  FP         0.282          0.291        0.079   3.561   0.123    0.427

In [14]:
# Path coefficients dan R²
model_sum$paths

          CA    FP
R^2    0.191 0.126
AdjR^2 0.178 0.117
IC     0.120 0.175
MO     0.364     .
EO     0.129     .
CA         . 0.282

In [15]:
# f² Effect Sizes
model_sum$fSquare

      IC    MO    EO    CA    FP
IC 0.000 0.000 0.000 0.017 0.032
MO 0.000 0.000 0.000 0.144 0.000
EO 0.000 0.000 0.000 0.020 0.000
CA 0.000 0.000 0.000 0.000 0.080
FP 0.000 0.000 0.000 0.000 0.000

## 5. Path Diagram

In [10]:
plot(pls_model)

<grViz HTML widget HTML Widget>

## 6. Interpretasi

### A. Evaluasi Model Pengukuran

**Konstruk Formatif (IC, MO, EO):**
- VIF semua indikator < 1.02 → tidak ada multikolinearitas (sangat baik).
- Outer weights perlu dievaluasi via bootstrap (signifikansi). Untuk formatif, loading rendah bisa diterima selama weight signifikan.

**Konstruk Reflektif (CA, FP):**
- rho_C: CA = 0.742, FP = 0.724 → marginal (≥ 0.70 ✅)
- AVE: CA = 0.423, FP = 0.397 → di bawah threshold 0.50. Ini artinya convergent validity kurang ideal, namun dalam konteks *training* dengan data dummy, kita tetap lanjutkan.
- Loading CA berkisar 0.54–0.78; FP berkisar 0.56–0.68.

### B. Pengujian Hipotesis (Bootstrap, n = 1000)

| # | Path | β | t-value | 95% CI | Keputusan |
|---|---|---|---|---|---|
| H1 | IC → CA | 0.120 | 1.474 | [−0.04, 0.28] | ❌ **Tidak Didukung** |
| H2 | MO → CA | 0.364 | 5.996 | [0.25, 0.49] | ✅ **Didukung*** |
| H3 | EO → CA | 0.129 | 1.859 | [0.01, 0.28] | ⚠️ **Marginal** (CI nyaris 0) |
| H4 | CA → FP | 0.282 | 3.561 | [0.12, 0.43] | ✅ **Didukung*** |
| H5 | IC → FP | 0.175 | 1.726 | [−0.04, 0.37] | ❌ **Tidak Didukung** |

### C. R² dan f²

| Variabel Endogen | R² | Adj. R² | Interpretasi |
|---|---|---|---|
| CA (Keunggulan Bersaing) | 0.191 | 0.178 | Lemah |
| FP (Kinerja Perusahaan) | 0.126 | 0.117 | Lemah |

| Path | f² | Efek |
|---|---|---|
| MO → CA | 0.144 | Sedang (mendekati medium) |
| CA → FP | 0.080 | Kecil–sedang |
| IC → FP | 0.032 | Kecil |
| IC → CA | 0.017 | Kecil |
| EO → CA | 0.020 | Kecil |

### D. Temuan Kunci

1. **Orientasi Pasar (MO)** adalah satu-satunya prediktor signifikan yang kuat terhadap Keunggulan Bersaing (β = 0.36, t = 6.0). Ini sejalan dengan teori *Market Orientation* (Narver & Slater) bahwa pemahaman pasar adalah kunci keunggulan.
2. **Kapabilitas Inovasi (IC)** tidak signifikan terhadap CA maupun FP — mengindikasikan bahwa inovasi saja tanpa orientasi pasar tidak cukup untuk meningkatkan kinerja.
3. **Keunggulan Bersaing → Kinerja** signifikan (β = 0.28) tapi R² FP hanya 12.6% — banyak faktor lain yang belum termasuk dalam model.
4. R² secara keseluruhan **lemah**, yang wajar untuk model eksploratori dengan sample kecil (n = 200). Dalam riset nyata, ini mengindikasikan perlunya tambahan prediktor.

**Catatan pedagogi:** Hasil ini sengaja menunjukkan bahwa **tidak semua hipotesis selalu didukung** — penting bagi mahasiswa untuk belajar menginterpretasi hasil non-signifikan.

## 7. Latihan

### Latihan 1: Tambah Path

Tambahkan path langsung `MO → FP` dan bandingkan R² FP.

```r
sm2 <- relationships(
  paths(from = c("IC", "MO", "EO"), to = "CA"),
  paths(from = c("CA", "IC", "MO"), to = "FP")  # MO ditambah
)
# pls_model2 <- estimate_pls(data = dat, measurement_model = mm, structural_model = sm2)
# summary(pls_model2)$paths
```

### Latihan 2: Variabel Kontrol

Masukkan `firm_age` sebagai variabel kontrol terhadap FP.

```r
mm2 <- constructs(
  composite("IC", multi_items("IC", 1:4), weights = mode_B),
  composite("MO", multi_items("MO", 1:3), weights = mode_B),
  composite("EO", multi_items("EO", 1:3), weights = mode_B),
  composite("CA", multi_items("CA", 1:4), weights = mode_A),
  composite("FP", multi_items("FP", 1:4), weights = mode_A),
  composite("firm_age", single_item("firm_age"))
)
sm2 <- relationships(
  paths(from = c("IC", "MO", "EO"), to = "CA"),
  paths(from = c("CA", "IC", "firm_age"), to = "FP")
)
# pls_ctrl <- estimate_pls(data = dat, measurement_model = mm2, structural_model = sm2)
# summary(pls_ctrl)
```

### Latihan 3: Specific Indirect Effects

Hitung efek tidak langsung IC → CA → FP menggunakan `specific_effect_significance()`.

```r
# ie <- specific_effect_significance(boot_model,
#   from = "IC", through = "CA", to = "FP")
# ie
```